## Exercise 4 

In [46]:
# we have to write a discrete event simulation program for a blocking system of m service units 
# and no waiting rooms. 
# this could be like a phone help service where there is no queue, if all lines are occupied
# the phone hangs up.

# The offered traffic A is the product of the mean arrival rate and the mean service time 
import numpy as np
import RNG_mat
import scipy.stats as stats
import math

np.random.seed(42)
# ------PART 1-------
# arrival process is poison. according to notes the diffrence between points in a poisson process 
# is exponential
N_CUSTUMERS = 10_000
M_SERVERS = 10
MU_SERVICE_TIME = 8
MU_ARRIVAL_TIME = 1

N_REPS = 10




def create_poison_dist_points(n_samples=10_000):
    # if Lambda is not None:
    #     mean_ = 1/Lambda
    arrival_intervals = np.random.exponential(scale=1/1, size=n_samples)
    return arrival_intervals

def create_poison_dist_points_service( n_samples=10_000):
    # if Lambda is not None:
    #     mean_ = 1/Lambda
    arrival_intervals = np.random.exponential(scale=8, size=n_samples)
    return arrival_intervals


def compute_ci(data, alpha=0.05):
    """Calculates the 95% Confidence Interval based on sub-samples."""
    n = len(data)
    mean = np.mean(data)
    std_err = np.std(data, ddof=1) / np.sqrt(n)
    t_crit = stats.t.ppf(1 - alpha/2, df=n - 1)
    return mean, mean - t_crit * std_err, mean + t_crit * std_err

def simulate_blocking(N_REPS,N_CUSTUMERS,arrival_distribution, service_distribution):
    blocked_list = []


    for rep in range(N_REPS):
        blocked_customers = 0
        arrival_intervals = arrival_distribution(n_samples=N_CUSTUMERS)
        arrival_times = np.cumsum(arrival_intervals)
        service_intervals = service_distribution( n_samples=N_CUSTUMERS)

        next_free_server = np.zeros((M_SERVERS))

        for i in range(N_CUSTUMERS):
            current_person_arrival = arrival_times[i]

            available_servers_mask = next_free_server<=current_person_arrival

            if np.any(available_servers_mask):
                idx = np.argmax(available_servers_mask) 
                next_free_server[idx] = current_person_arrival + service_intervals[i]
            else: 
                # no free cashier
                blocked_customers +=1

        blocked_list.append(blocked_customers/N_CUSTUMERS)
    
    return blocked_list

blocked_list = simulate_blocking(N_REPS,N_CUSTUMERS, create_poison_dist_points,create_poison_dist_points_service)
mean, lower, upper = compute_ci(blocked_list)

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

def calculate_erlingB(A, M_SERVERS):
    m_list = np.arange(0,M_SERVERS+1,1).astype(int)
    bot = [math.factorial(m_list[i]) for i in range (len(m_list))]
    B = (A**M_SERVERS/math.factorial(M_SERVERS)) / (np.sum(A**m_list/ (np.array(bot))))
    return B
B = calculate_erlingB(A, M_SERVERS)

print(f"Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")


Simulated blocking function mean: 0.1233, Confidence 95%: [0.1176, 0.129] 
Exact Erlang-B from slides: 0.1217 


In [ ]:
# Part 2 


def erlang_distribution( n_samples):
    return np.random.gamma(shape=2, scale=1/2, size=n_samples)

def hyper_exp_distribution(n_samples):
    p1, lam1, lam2 = 0.8 , 0.8333, 5.0
    RN = np.random.rand(n_samples)
    mask = RN <= p1
    default = np.random.exponential(1/lam1, size=n_samples)
    second = np.random.exponential(1/lam2, size=n_samples)
    return np.where(mask, default, second)


# a)  
blocked_list = simulate_blocking(N_REPS,N_CUSTUMERS,erlang_distribution, create_poison_dist_points_service)
mean, lower, upper = compute_ci(blocked_list )

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

print(f"a) Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")

# b)
blocked_list = simulate_blocking(N_REPS,N_CUSTUMERS,hyper_exp_distribution, create_poison_dist_points_service)
mean, lower, upper = compute_ci(blocked_list )

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

print(f"\nb) Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")

a) Simulated blocking function mean: 0.0941, Confidence 95%: [0.0894, 0.0989] 
Exact Erlang-B from slides: 0.1217 

b) Simulated blocking function mean: 0.1359, Confidence 95%: [0.1315, 0.1403] 
Exact Erlang-B from slides: 0.1217 


In [58]:
# task 3 

# a)  constant serving time minutes =8

def constant_serving_time(n_samples):
    return np.ones((n_samples))*8


blocked_list = simulate_blocking(N_REPS,N_CUSTUMERS,create_poison_dist_points, constant_serving_time)
mean, lower, upper = compute_ci(blocked_list )

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

print(f"a) Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")

# b) pareto distribution 
def pareto_distribution(n_samples):
    k = 1.5
    beta = MU_SERVICE_TIME * (k - 1.0) / k
    return (np.random.pareto(k, size=n_samples)+1)*beta



blocked_list = simulate_blocking(N_REPS,N_CUSTUMERS,create_poison_dist_points, pareto_distribution)
mean, lower, upper = compute_ci(blocked_list )

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

print(f"\nb) Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")


# c) normal dist (own chosen)
def normal_distribution(n_samples):
    return np.random.uniform(low = 0, high=2.0*MU_SERVICE_TIME, size=n_samples)

blocked_list = simulate_blocking(N_REPS,N_CUSTUMERS,create_poison_dist_points, normal_distribution)
mean, lower, upper = compute_ci(blocked_list )

A = MU_SERVICE_TIME* 1/MU_ARRIVAL_TIME

print(f"\nc) Simulated blocking function mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")
print(f"Exact Erlang-B from slides: {np.round(B,4)} ")


a) Simulated blocking function mean: 0.1188, Confidence 95%: [0.1165, 0.1211] 
Exact Erlang-B from slides: 0.1217 

b) Simulated blocking function mean: 0.1088, Confidence 95%: [0.1007, 0.1169] 
Exact Erlang-B from slides: 0.1217 

c) Simulated blocking function mean: 0.1242, Confidence 95%: [0.1188, 0.1297] 
Exact Erlang-B from slides: 0.1217 


In [ ]:
# part 4 